# Week 3: Model evaluation

Topics: simple baselines, data leakage, 5-fold cross-validation, and confusion matrices. Small numeric examples come first, followed by the housing dataset.

Run from the `ml` folder with `numpy`, `pandas`, and `scikit-learn` installed.

## 1. Classification baseline

Six houses have class labels: `1` means high priced and `0` means lower priced. Compare a model's predictions with an always-high prediction. Both accuracy values are calculated below.

In [1]:
from sklearn.metrics import accuracy_score, confusion_matrix
actual =    [1, 1, 1, 1, 0, 0]
predicted = [1, 1, 0, 1, 0, 1]
always_one = [1, 1, 1, 1, 1, 1]
print('Model accuracy:', accuracy_score(actual, predicted))
print('Always-high baseline accuracy:', accuracy_score(actual, always_one))
print('Model confusion matrix (rows=actual, columns=predicted):')
print(confusion_matrix(actual, predicted, labels=[0, 1]))

Model accuracy: 0.6666666666666666
Always-high baseline accuracy: 0.6666666666666666
Model confusion matrix (rows=actual, columns=predicted):
[[1 1]
 [1 3]]


The model correctly classifies 4 of 6 houses, giving accuracy **0.667**. Always predicting class `1` also gets 4 of 6 correct. Equal accuracy means the model has not improved on this simple baseline. The model's errors are the third and sixth houses.

## 2. Regression baseline

A price baseline predicts the **mean training price** for every test house. Mean absolute error (MAE) gives the average prediction error in euros. A lower MAE is better.

In [2]:
import numpy as np
from sklearn.metrics import mean_absolute_error
train_prices = np.array([200_000, 300_000, 400_000])
test_prices = np.array([250_000, 350_000])
baseline_guesses = np.full(len(test_prices), train_prices.mean())
model_guesses = np.array([260_000, 330_000])
print('Baseline guess:', baseline_guesses[0])
print('Baseline MAE:', mean_absolute_error(test_prices, baseline_guesses))
print('Model MAE:', mean_absolute_error(test_prices, model_guesses))

Baseline guess: 300000.0
Baseline MAE: 50000.0
Model MAE: 15000.0


The baseline mean comes from training prices only. Using test prices to calculate that mean would reveal information from the held-out answers. Here the model's MAE is **€15,000**, compared with the baseline's **€50,000**.

## 3. Data leakage

A prediction input must be available **at prediction time**. `high_price` means `sale_price_eur >= €500,000`. Including `sale_price_eur` in the classifier would reveal the target. The same applies to a column calculated from sale price, such as `sale_price_per_sqm`. Living area is available before a sale and is a valid input.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path('data/house_pricing_ml_exercise.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Run from the ml folder; data/house_pricing_ml_exercise.csv is required.')

numeric_features = [
    'bedrooms', 'bathrooms', 'living_area_sqm', 'lot_size_sqm', 'year_built',
    'parking_spaces', 'distance_city_center_km', 'school_rating_1_10',
    'crime_rate_1_10', 'has_garden', 'has_balcony',
]
categorical_features = ['city', 'neighborhood', 'property_type', 'condition', 'energy_rating']


def load_housing():
    df = pd.read_csv(DATA_PATH, na_values=['?', 'n/a', 'NA', ''])
    # Remove repeated properties before splitting, so copies cannot appear in both sets.
    df = df.drop_duplicates(subset=[c for c in df.columns if c != 'house_id']).copy()
    for col in numeric_features[:-2]:
        # Handles entries such as "1 baths" and "90m2"; invalid entries become NaN.
        numbers = df[col].astype('string').str.extract(r'(-?\d+(?:\.\d+)?)', expand=False)
        df[col] = pd.to_numeric(numbers, errors='coerce')
    for col in ['has_garden', 'has_balcony']:
        values = df[col].astype('string').str.strip().str.lower()
        df[col] = values.map({'yes': 1, 'y': 1, 'true': 1, '1': 1,
                              'no': 0, 'n': 0, 'false': 0, '0': 0}).astype(float)
    for col in categorical_features:
        clean = df[col].astype('string').str.strip().str.lower()
        df[col] = clean.astype(object).where(clean.notna(), np.nan)
    df['city'] = df['city'].replace({
        'berln': 'berlin', 'colgne': 'cologne', 'koeln': 'cologne',
        'frankfrut': 'frankfurt', 'hambug': 'hamburg',
        'leipzg': 'leipzig', 'münich': 'munich',
    })
    df['property_type'] = df['property_type'].replace({
        'apartmnt': 'apartment', 'detached': 'detached house',
        'detatched house': 'detached house', 'semi detached': 'semi-detached',
        'semi-detatched': 'semi-detached', 'terrace house': 'terraced house',
        'terraced hse': 'terraced house',
    })
    df['condition'] = df['condition'].replace({
        'averge': 'average', 'excelent': 'excellent', 'gud': 'good',
        'needs renovaton': 'needs renovation', 'renovation needed': 'needs renovation',
    })
    df['energy_rating'] = df['energy_rating'].replace({'dd': 'd'})
    df['sale_price_eur'] = pd.to_numeric(df['sale_price_eur'], errors='coerce')
    df['high_price'] = pd.to_numeric(df['high_price'], errors='coerce')
    return df.dropna(subset=['sale_price_eur', 'high_price']).reset_index(drop=True)

housing = load_housing()
print(f'{len(housing)} houses; {housing.high_price.mean():.1%} are high priced')
print('Target matches EUR 500,000 rule:',
      bool(((housing.sale_price_eur >= 500_000).astype(int) == housing.high_price).all()))

1200 houses; 63.4% are high priced
Target matches EUR 500,000 rule: True


In [4]:
safe_features = numeric_features + categorical_features
X = housing[safe_features]
y = housing['high_price'].astype(int)
assert 'sale_price_eur' not in X.columns and 'high_price' not in X.columns
print('Pre-sale feature count:', len(safe_features))

Pre-sale feature count: 16


## 4. Final test set

Reserve 20% of houses for a final check. Model selection uses only the remaining training houses. `stratify=y` keeps the high-price share similar in both sets.

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Training houses:', len(X_train), '| Final test houses:', len(X_test))
print('High-price share: train', round(y_train.mean(), 3),
      '| test', round(y_test.mean(), 3))

Training houses: 960 | Final test houses: 240
High-price share: train 0.634 | test 0.633


## 5. Five-fold cross-validation

Split the training rows into five groups, or *folds*. Fit on four folds and evaluate on the fifth. Repeat until each fold has served as the validation fold once. Five scores show how much performance varies across splits.

The small example below lists the fitting and validation row positions in each round.

In [6]:
from sklearn.model_selection import KFold
for round_number, (train, check) in enumerate(KFold(n_splits=5).split(range(10)), start=1):
    print(f'Round {round_number}: learn from {list(train)}; check {list(check)}')

Round 1: learn from [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]; check [np.int64(0), np.int64(1)]
Round 2: learn from [np.int64(0), np.int64(1), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]; check [np.int64(2), np.int64(3)]
Round 3: learn from [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]; check [np.int64(4), np.int64(5)]
Round 4: learn from [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(8), np.int64(9)]; check [np.int64(6), np.int64(7)]
Round 5: learn from [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]; check [np.int64(8), np.int64(9)]


Each row appears in a validation fold **once**. Results across five folds show sensitivity to the data split. For classification, **stratified** folds keep approximately the same high-price share in every fold.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_preprocessor(numeric_columns=numeric_features, categorical_columns=categorical_features):
    return ColumnTransformer([
        ('numbers', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), numeric_columns),
        ('categories', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_columns),
    ])

## 6. Cross-validate three models

Compare a majority-class baseline, logistic regression, and random forest. In each fold, the pipeline learns imputation and encoding from that fold's fitting rows. Mean F1 is used for model selection because it considers both precision and recall.

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    'Majority baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic regression': Pipeline([
        ('prepare', make_preprocessor()), ('model', LogisticRegression(max_iter=2000))
    ]),
    'Random forest': Pipeline([
        ('prepare', make_preprocessor()),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1))
    ]),
}
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=folds,
                            scoring={'accuracy': 'accuracy', 'f1': 'f1'}, n_jobs=1)
    cv_rows.append({
        'Model': name,
        'Mean accuracy': scores['test_accuracy'].mean(),
        'Mean F1': scores['test_f1'].mean(),
        'F1 spread (SD)': scores['test_f1'].std(),
    })
cv_results = pd.DataFrame(cv_rows).set_index('Model')
display(cv_results.round(3))
best_name = cv_results['Mean F1'].idxmax()
print('Selected using training folds:', best_name)

,Mean accuracy,Mean F1,F1 spread (SD)
Model,,,
Majority baseline,0.634,0.776,0.002
Logistic regression,0.922,0.938,0.010
Random forest,0.882,0.907,0.005


Selected using training folds: Logistic regression


The cross-validation table shows mean accuracy, mean F1, and the spread of F1 across folds. A model improves on the baseline when its mean score is higher. A small spread suggests more consistent scores across these folds but does not guarantee future performance.

## 7. Final test evaluation

Fit the model selected by cross-validation on all training rows, then evaluate it on the untouched test rows. The final test score did not determine the model choice.

In [9]:
from sklearn.metrics import confusion_matrix
best_model = models[best_name]
best_model.fit(X_train, y_train)
final_pred = best_model.predict(X_test)
print('Test accuracy:', round(accuracy_score(y_test, final_pred), 3))
print('Test precision:', round(precision_score(y_test, final_pred, zero_division=0), 3))
print('Test recall:', round(recall_score(y_test, final_pred, zero_division=0), 3))
print('Test F1:', round(f1_score(y_test, final_pred, zero_division=0), 3))
cm = confusion_matrix(y_test, final_pred, labels=[0, 1])
display(pd.DataFrame(cm,
    index=['Actually low (0)', 'Actually high (1)'],
    columns=['Predicted low (0)', 'Predicted high (1)']))
print(f'False positives: {cm[0,1]} | False negatives: {cm[1,0]}')

Test accuracy: 0.933
Test precision: 0.947
Test recall: 0.947
Test F1: 0.947


,Predicted low (0),Predicted high (1)
Actually low (0),80,8
Actually high (1),8,144


False positives: 8 | False negatives: 8


## 8. Error types

A **false positive** is a lower-priced house predicted high. A **false negative** is a high-priced house predicted low. Both counts appear in the confusion matrix; accuracy combines them into a single number.

The preferred balance between false positives and false negatives depends on how predictions will be used. Changing the probability threshold changes that balance.

## Summary

- A baseline shows whether a trained model improves on a simple guess.
- `sale_price_eur` and values calculated from it leak the high-price target.
- In 5-fold cross-validation, each training row serves as a validation row once.
- False positives occupy the upper-right confusion-matrix cell; false negatives occupy the lower-left.
- Model selection uses training folds. The held-out test set provides a final check.